# 🌳 Clustering Jerárquico (Agglomerative)
> **Caso de negocio:** Banco / Financiera Efectiva · Exploración de segmentos de clientes
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

El equipo de CRM de Financiera Efectiva quiere explorar la **estructura natural** de su base de clientes a partir de su comportamiento (recencia, frecuencia, ticket promedio, categorías compradas, antigüedad y uso de canal digital) — **sin decidir de antemano cuántos segmentos existen**.

A diferencia de K-Means (que exige fijar `k` desde el inicio), el clustering jerárquico construye un **dendrograma**: un árbol de fusiones sucesivas que permite decidir el número de cortes (segmentos) observando la estructura de los datos.

**Objetivo:** Construir el dendrograma de la base de clientes, elegir un número de segmentos razonable, y traducir los clusters técnicos en una taxonomía de negocio.

**KPIs de éxito:**
- Coeficiente cofenético >= 0.60 (el dendrograma preserva razonablemente las distancias originales)
- Silhouette score >= 0.30
- Segmentos interpretables: cada cluster debe tener un perfil de comportamiento claro

**Algoritmo:** Agglomerative Clustering (linkage='ward') + dendrograma (scipy)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q scipy plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage as sp_linkage, cophenet
from scipy.spatial.distance import pdist

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Aprendizaje no supervisado — clustering jerárquico',
    'features': [
        'recencia_dias       → días desde la última compra',
        'frecuencia_compras  → número de compras en el período de análisis',
        'ticket_prom_k       → ticket promedio (miles de S/.)',
        'n_categorias        → número de categorías de producto compradas',
        'meses_cliente       → antigüedad como cliente (meses)',
        'canal_digital       → 1 si usa canal digital (app/web), 0 si no',
    ],
    'criterios_aceptacion': {
        'coeficiente_cofenetico': '>= 0.60',
        'Silhouette':              '>= 0.30',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con 4 perfiles de comportamiento de cliente
# En producción: reemplazar con agregación de transacciones + CRM (BigQuery)
perfiles = {
    'vip':       {'rec': (7, 4),   'frec': (20, 4), 'tick': (12, 3),  'cats': (5, 1), 'ant': (36, 8), 'dig': 0.90},
    'regular':   {'rec': (30, 12), 'frec': (8, 3),  'tick': (5, 2),   'cats': (3, 1), 'ant': (18, 6), 'dig': 0.60},
    'en_riesgo': {'rec': (90, 20), 'frec': (3, 2),  'tick': (3, 1.5), 'cats': (2, 1), 'ant': (24, 8), 'dig': 0.35},
    'nuevo':     {'rec': (10, 5),  'frec': (3, 2),  'tick': (4, 2),   'cats': (2, 1), 'ant': (4, 2),  'dig': 0.75},
}

N_POR_PERFIL = 75
rows = []
for seg, pm in perfiles.items():
    for _ in range(N_POR_PERFIL):
        rows.append({
            'recencia_dias':      max(1, int(np.random.normal(*pm['rec']))),
            'frecuencia_compras': max(1, int(np.random.normal(*pm['frec']))),
            'ticket_prom_k':      max(0.5, round(np.random.normal(*pm['tick']), 1)),
            'n_categorias':       max(1, int(np.random.normal(*pm['cats']))),
            'meses_cliente':      max(1, int(np.random.normal(*pm['ant']))),
            'canal_digital':      int(np.random.random() < pm['dig']),
            'segmento_real':      seg,
        })

df = pd.DataFrame(rows)
df.insert(0, 'cliente_id', range(1, len(df)+1))
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset: {df.shape}')
df.describe().round(2)

In [ ]:
# Distribución de las variables de comportamiento
FEATURES = ['recencia_dias', 'frecuencia_compras', 'ticket_prom_k', 'n_categorias', 'meses_cliente', 'canal_digital']

fig = make_subplots(rows=2, cols=3, subplot_titles=FEATURES)
for idx, feat in enumerate(FEATURES):
    r, c = idx // 3 + 1, idx % 3 + 1
    fig.add_trace(go.Histogram(x=df[feat], marker_color='#1a4c8c', nbinsx=20, showlegend=False), row=r, col=c)

fig.update_layout(height=500, title='Distribución de variables de comportamiento',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Recencia vs frecuencia, coloreado por ticket promedio (sin clusters aún)
fig = px.scatter(df, x='recencia_dias', y='frecuencia_compras', color='ticket_prom_k',
                 color_continuous_scale='teal', opacity=0.6,
                 title='Recencia vs Frecuencia (color = ticket promedio)')
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
X_raw = df[FEATURES].fillna(0).values

# El clustering jerárquico con distancia euclidiana es sensible a la escala — estandarizar es obligatorio
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Observaciones: {X.shape[0]} | Features: {X.shape[1]}')
print(f'Medias después de escalar: {X.mean(axis=0).round(3)}')
print(f'Desv. estándar después de escalar: {X.std(axis=0).round(3)}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Matriz de enlace (linkage) con método 'ward' — minimiza la varianza intra-cluster
Z = sp_linkage(X, method='ward')

# Dendrograma truncado (últimas 30 fusiones) para una lectura clara
fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=8, color_threshold=Z[-4, 2], ax=ax)
ax.set_title('Dendrograma (truncado a las últimas 30 fusiones)')
ax.set_xlabel('Clientes agrupados (o tamaño de grupo entre paréntesis)')
ax.set_ylabel('Distancia (ward)')
ax.axhline(y=Z[-4, 2], color='#c0392b', linestyle='--', label='Corte en 4 segmentos')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative Clustering con 4 segmentos (definidos a partir del corte del dendrograma)
N_CLUSTERS = 4
model = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
labels = model.fit_predict(X)

df['cluster'] = labels
df['cluster_label'] = [f'Cluster {l+1}' for l in labels]

print(f'Segmentos encontrados: {N_CLUSTERS}')
print(f'\nTamaño de clusters:')
print(df['cluster_label'].value_counts())

## 5️⃣ Fase 5 — Evaluation

In [ ]:
sil = silhouette_score(X, labels)
ch  = calinski_harabasz_score(X, labels)
db  = davies_bouldin_score(X, labels)
coph, _ = cophenet(Z, pdist(X))

metrics = {'Coeficiente cofenetico': coph, 'Silhouette': sil, 'Calinski-Harabasz': ch, 'Davies-Bouldin': db}
criterios = {'Coeficiente cofenetico': 0.60, 'Silhouette': 0.30}

print('=== MÉTRICAS DE CLUSTERING ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral >= {umbral})'
    print(f'  {k:24s}: {v:.4f}  {estado}')

In [ ]:
# Perfil medio por cluster (en escala original)
profiles = df.groupby('cluster_label')[FEATURES].mean().round(2)
profiles['n'] = df['cluster_label'].value_counts()
profiles = profiles.reset_index()

colors = ['#1a4c8c', '#0d9488', '#c0392b', '#d97706']
fig = make_subplots(rows=1, cols=len(FEATURES), subplot_titles=FEATURES)
for j, feat in enumerate(FEATURES):
    fig.add_trace(go.Bar(x=profiles['cluster_label'], y=profiles[feat],
                          marker_color=colors, showlegend=False), row=1, col=j+1)

fig.update_layout(height=380, title='Perfil medio por cluster',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()
display(profiles)

In [ ]:
# Validación: ¿los clusters recuperan los perfiles reales del DGP?
cross = pd.crosstab(df['cluster_label'], df['segmento_real'], normalize='index')

fig = px.imshow(cross, text_auto='.0%', color_continuous_scale='teal',
                title='Composición de cada cluster por perfil real (% por fila)')
fig.update_layout(height=380, paper_bgcolor='white')
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Etiquetado de negocio: ordenar clusters por valor (recencia baja + frecuencia/ticket/categorías/antigüedad altas = mejor)
profiles['score'] = (
    -profiles['recencia_dias'].rank()
    + profiles['frecuencia_compras'].rank()
    + profiles['ticket_prom_k'].rank()
    + profiles['meses_cliente'].rank()
)
profiles_sorted = profiles.sort_values('score', ascending=False).reset_index(drop=True)

etiquetas_negocio = ['VIP', 'Regular', 'En riesgo', 'Nuevo']
profiles_sorted['segmento_negocio'] = etiquetas_negocio[:len(profiles_sorted)]

mapa_segmento = dict(zip(profiles_sorted['cluster_label'], profiles_sorted['segmento_negocio']))
df['segmento_negocio'] = df['cluster_label'].map(mapa_segmento)

print('Mapeo cluster → segmento de negocio:')
display(profiles_sorted[['cluster_label', 'n'] + FEATURES + ['segmento_negocio']])

print('\nDistribución final de segmentos:')
print(df['segmento_negocio'].value_counts())

In [ ]:
# Guardar outputs
df[['cliente_id'] + FEATURES + ['segmento_negocio']].to_csv('clientes_segmentados_efectiva.csv', index=False)
print('Archivo clientes_segmentados_efectiva.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'scaler': scaler, 'features': FEATURES, 'mapa_segmento': mapa_segmento,
            'linkage_matrix': Z}, 'modelo_hierarchical_efectiva.pkl')
print('Modelo modelo_hierarchical_efectiva.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_clientes = len(df)
vip = (df['segmento_negocio'] == 'VIP').sum()
en_riesgo = (df['segmento_negocio'] == 'En riesgo').sum()

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes analizados:                {n_clientes:,}')
print(f'Segmentos identificados:            {N_CLUSTERS}')
print(f'VIP:                                 {vip:,} ({vip/n_clientes:.0%}) → programa de fidelización')
print(f'En riesgo:                           {en_riesgo:,} ({en_riesgo/n_clientes:.0%}) → campaña de retención')
print(f'Coeficiente cofenético:              {coph:.3f}')
print(f'Silhouette score:                    {sil:.3f}')
print(f'\nArquitectura de producción:')
print('  Transacciones + CRM → BigQuery → recalculo trimestral de segmentos → taxonomía de clientes para campañas')